# Oh C-semantics surface inversion evaluation on UAVSAR C3 data

This notebook runs the legacy Oh surface inversion with C-style intermediate precision on the UAVSAR multilooked C3 product, then compares the Python result with the C-PolSARpro output. It is the C-semantics counterpart of `test-oh-surface-inversion-real-data-uavsar.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

from polsarpro.dev.devtools import parse_psp_parameter_string
from polsarpro.dev.io import read_psp_bin
from polsarpro.dev.metrics import summarize_metrics, visualize_errors
from polsarpro.io import open_netcdf_beam
from polsarpro.physical_inversion import oh_surface_inversion
from polsarpro.util import pauli_rgb

# Change this to your local C-PolSARpro installation directory.
c_psp_dir = Path("/home/c_psp/Soft/bin")
os.environ["PATH"] += os.pathsep + str(c_psp_dir / "data_process_sngl")
os.environ["PATH"] += os.pathsep + str(c_psp_dir / "data_convert")
# The C Newton-loop counter is shared between OpenMP workers.
os.environ["OMP_NUM_THREADS"] = "1"

product_id = "winnip_09002_12060_004_120714_L090_CX_02"

input_uavsar_nc = Path(f"/data/psp/test_files/{product_id}_ML5X5.nc")
input_test_dir = Path("/data/psp/winnip_cpsp/C3")
output_test_dir = Path("/data/psp/res/oh_c3")
output_test_dir.mkdir(parents=True, exist_ok=True)

incidence_angle_file = Path(f"/data/psp/test_files/{product_id}.inc")
incidence_angle_c_file = Path(
    "/data/psp/winnip_cpsp/incidence_angle_ml5x5_rad.bin"
)
file_out = Path("/data/psp/res/test_oh_surface_inversion_c_semantics_comparison_uavsar.nc")

# The C implementation uses -th1 for HV/VV and -th2 for HH/VV. Its
# command-line help displays those descriptions in the opposite order.
thresh1_db = -5.0  # HV/VV
thresh2_db = 0.0  # HH/VV
angle_unit = 1  # C-PolSARpro: 0 = degrees, 1 = radians

## Load UAVSAR data and incidence-angle raster

In [ ]:
C3 = open_netcdf_beam(input_uavsar_nc)
row_dim, col_dim = C3.dims

# Dimensions recorded in the UAVSAR annotation file:
# hgt.set_rows (pixels) = 3767
# hgt.set_cols (pixels) = 11636
nlines = 3767
ncols = 11636

# The incidence-angle file is a little-endian float32 stream in radians.
# Read its raster-sized prefix and reshape it to the full-resolution grid.
incidence_angle_values = np.fromfile(
    incidence_angle_file,
    dtype="<f4",
    count=nlines * ncols,
)
incidence_angle_values[incidence_angle_values == -10000] = np.nan
incidence_angle = xr.DataArray(
    incidence_angle_values.reshape(nlines, ncols),
    dims=(row_dim, col_dim),
    name="incidence_angle",
).coarsen(y=5, x=5, boundary="trim").mean()

# Write the multilooked incidence-angle stream used by C-PolSARpro.
incidence_angle.astype(np.float32).values.tofile(incidence_angle_c_file)

## Run the C version on the C3 input

The threshold values are passed directly to `-th1` and `-th2`, following the C implementation rather than the reversed descriptions printed by its help text.

In [ ]:
fnr, fnc = C3.sizes[row_dim], C3.sizes[col_dim]

input_str = f"""id: {input_test_dir}
od: {output_test_dir}
iodf: C3
ang: {incidence_angle_c_file}
ofr: 0
ofc: 0
fnr: {fnr}
fnc: {fnc}
un: {angle_unit}
th1: {thresh1_db}
th2: {thresh2_db}
errf: /tmp"""

parameters = parse_psp_parameter_string(input_str)
os.system(f"surface_inversion_oh.exe {parameters}")

# C-PolSARpro does not create another config file in the output directory.
shutil.copy(input_test_dir / "config.txt", output_test_dir)

## Apply the Python surface inversion with C semantics

In [ ]:
with ProgressBar():
    oh_surface_inversion(
        C3,
        incidence_angle=incidence_angle,
        thresh1=thresh1_db,
        thresh2=thresh2_db,
        c_semantics=True,
    ).to_netcdf(file_out, mode="w")

## Inspect results

In [ ]:
out_py = xr.open_dataset(file_out).load()
out_py.close()
out_py

## Numerical evaluation

In [ ]:
out_c = {}
out_names = (
    "oh_er",
    "oh_mv",
    "oh_ks",
    "oh_mask_out",
    "oh_mask_in",
    "oh_mask_valid_in_out",
)
for name in out_names:
    out_c[name] = read_psp_bin(output_test_dir / f"{name}.bin").T

out_py = out_py.transpose("x", "y")
summarize_metrics(out_py, out_c, short_titles=False, verbose=False)

In [ ]:
visualize_errors(out_py=out_py, out_c=out_c, sub_az=1, sub_rg=1)

## Spatial overview

In [ ]:
rgb = pauli_rgb(C3).transpose("x", "y", "band")

plt.figure(figsize=(15, 10))
plt.subplot(1, 2, 1)
plt.imshow(rgb, interpolation="none")
plt.title("Pauli RGB")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(out_py.oh_mask_valid_in_out, interpolation="none")
plt.title("Oh combined validity mask")
plt.axis("off")
plt.tight_layout()